# 三步质控快速版

奇偶深度校验 → 行星半径合理性 → NASA 档案交叉比对，**一步完成即时写 CSV**。

**提速手段**（相对分开运行的三个 notebook）：
- 奇偶校验：BLS 降分辨率 + 多扇区拼接 + 下载缓存
- 半径校验：TIC 星表**批量查询**（一次 `query_criteria` 替代 N 次 `query_object`）
- 交叉比对：缓存 + 名称/TIC ID 双键

**输出**：`vetting_result.csv`，列结构与之前 `nasa_crossmatch_result.csv` 完全一致。

## 0. 安装依赖

In [1]:
%pip install -q numpy pandas astropy astroquery lightkurve

Note: you may need to restart the kernel to use updated packages.


## 1. 配置与加载数据

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 260)

# ==================== 参数配置 ====================
ODD_EVEN_THRESHOLD   = 3.0          # σ > 3 -> 奇偶不通过
MAX_RP_GENERAL       = 2.5          # Rjup，通用行星半径上限
MAX_RP_MDWARF        = 2.0          # Rjup，M 矮星更严上限
R_SUN_TO_RJUP        = 9.7311       # 1 Rsun = 9.73 Rjup
R_JUP_TO_REARTH      = 11.2089      # 1 Rjup = 11.21 Rearth
STITCH_ALL_SECTORS   = True         # 拼接多扇区 -> 更多凌日
TESS_CACHE_DIR       = 'tess_cache'
OUTPUT_CSV           = 'outputs/output.csv'

os.makedirs(TESS_CACHE_DIR, exist_ok=True)

# ==================== 输入 ====================
_PATHS = ['inputs/put_files_here/input.csv',
          '02.csv',
          'inputs/ignore_these/01.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[0])

df = pd.read_csv(INPUT_CSV)
ORIG_DF = df.copy()                          # Person1 原始快照，末尾校验用

# Person1 的 8 个原始列
P1_COLS = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
           'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']

# 最终输出列顺序（与 nasa_crossmatch_result.csv 一致）
FINAL_COLS = P1_COLS + [
    'depth_odd_ppm', 'depth_even_ppm', 'n_transits',
    'odd_even_sigma', 'odd_even_reason', 'odd_even_pass',
    'TIC_ID', 'Stellar_Radius_Rsun', 'Teff_K', 'Spectral_Type',
    'Rp_Rjup', 'Rp_Rearth', 'radius_reason', 'planet_radius_plausible',
    '已知/新候选', 'match_source']

print(f'已加载 {INPUT_CSV}  ->  {len(df)} 个候选')
df.head()

已加载 outputs/confirmed_100.csv  ->  29 个候选


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive
0,Kepler-528,40,8.25305,0.21041,10031.3,1.10,5.58,True
1,TOI-5076,70,5.58267,0.13242,439.2,0.50,3.73,True
2,TOI-2580,59,3.39686,0.00085,2841.2,4.15,48.09,False
3,Kepler-1254,80,11.90309,0.31842,14136.2,1.70,5.59,True
4,TOI-1410,16,1.21631,0.00011,1082.3,1.10,21.18,False


## 2. 校验一：奇偶深度检验（快速版）

In [3]:
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

_LC_CACHE = {}

def _get_lc(target, sector):
    '''下载光变曲线(带缓存)，优先 SPOC + 多扇区拼接。'''
    key = (str(target), int(sector))
    if key in _LC_CACHE:
        return _LC_CACHE[key]
    search = None
    for kw in ({'sector': int(sector), 'author': 'SPOC'},
               {'author': 'SPOC'},
               {'sector': int(sector)},
               {}):
        try:
            sr = lk.search_lightcurve(str(target), **kw)
            if sr is not None and len(sr) > 0:
                search = sr
                break
        except Exception:
            continue
    if search is None:
        _LC_CACHE[key] = None
        return None
    lc = None
    try:
        if STITCH_ALL_SECTORS and len(search) > 1:
            lc = search.download_all(download_dir=TESS_CACHE_DIR).stitch()
        else:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        try:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
        except Exception:
            lc = None
    _LC_CACHE[key] = lc
    return lc

def odd_even_check(target, sector, period, duration_hr):
    '''返回 {depth_odd, depth_even, n_transits, sigma, reason}。'''
    out = {'depth_odd': np.nan, 'depth_even': np.nan, 'n_transits': 0,
           'sigma': np.nan, 'reason': ''}
    lc = _get_lc(target, sector)
    if lc is None:
        out['reason'] = 'no_lc'
        return out
    try:
        lc = lc.remove_nans().normalize().flatten(window_length=401)
        t = np.asarray(lc.time.value, dtype=float)
        f = np.asarray(lc.flux.value, dtype=float)
        g = np.isfinite(t) & np.isfinite(f)
        t, f = t[g], f[g]
        if len(t) < 200:
            out['reason'] = f'few_cad({len(t)})'
            return out
        mad = np.nanmedian(np.abs(f - np.nanmedian(f)))
        scl = 1.4826 * mad if mad > 0 else np.nanstd(f)
        dy = np.full(len(t), scl if scl > 0 else 1e-3)
        bls = BoxLeastSquares(t, f, dy=dy)
        p0 = float(period)
        ps = np.linspace(p0 * 0.999, p0 * 1.001, 300)
        d0 = max(float(duration_hr), 0.5) / 24.0
        ds = np.unique(np.clip(np.array([d0 * 0.6, d0, d0 * 1.4]),
                               0.5 / 24.0, 8.0 / 24.0))
        res = bls.power(ps, ds, objective='snr')
        j = int(np.argmax(res.power))
        st = bls.compute_stats(float(res.period[j]),
                              float(res.duration[j]),
                              float(res.transit_time[j]))
        ntr = int(np.sum(np.atleast_1d(st['per_transit_count']) > 0))
        out['n_transits'] = ntr
        do, eo = float(st['depth_odd'][0]), float(st['depth_odd'][1])
        de, ee = float(st['depth_even'][0]), float(st['depth_even'][1])
        out['depth_odd'] = do * 1e6
        out['depth_even'] = de * 1e6
        comb = np.sqrt(eo ** 2 + ee ** 2)
        if not np.isfinite(comb) or comb <= 0:
            out['reason'] = f'err_undef(n={ntr})'
            return out
        out['sigma'] = abs(do - de) / comb
        out['reason'] = 'ok'
        return out
    except Exception as e:
        out['reason'] = f'ex:{type(e).__name__}'
        return out

In [4]:
print('=== 校验一：奇偶深度 ===')
ov = [(np.nan, np.nan, 0, np.nan, 'skipped')] * len(df)
for i, row in df.iterrows():
    r = odd_even_check(row['Target'], row['Sector'],
                       row['Period (d)'], row['Duration (hr)'])
    ov[i] = (r['depth_odd'], r['depth_even'], r['n_transits'], r['sigma'], r['reason'])
    tag = 'NaN' if not np.isfinite(r['sigma']) else f"{r['sigma']:.2f}"
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} "
          f"sigma={tag:<6} n={r['n_transits']:<3} {r['reason']}")

d1, d2, ntr, sig, rsn = zip(*ov)
df['depth_odd_ppm'] = d1
df['depth_even_ppm'] = d2
df['n_transits'] = ntr
df['odd_even_sigma'] = sig
df['odd_even_reason'] = rsn
df['odd_even_pass'] = pd.array(
    [(x <= ODD_EVEN_THRESHOLD) if np.isfinite(x) else pd.NA for x in sig],
    dtype='boolean')

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   通过: {int((df["odd_even_pass"]==True).sum())}  '
      f'不通过: {int((df["odd_even_pass"]==False).sum())}  '
      f'无法判定: {int(df["odd_even_pass"].isna().sum())}')

=== 校验一：奇偶深度 ===
[  1/29] Kepler-528         sigma=1.11   n=3   ok
[  2/29] TOI-5076           sigma=0.00   n=4   ok
[  3/29] TOI-2580           sigma=6.46   n=7   ok
[  4/29] Kepler-1254        sigma=1.35   n=2   ok
[  5/29] TOI-1410           sigma=0.86   n=16  ok
[  6/29] Kepler-224         sigma=1.11   n=5   ok
[  7/29] Kepler-1546        sigma=1.98   n=6   ok
[  8/29] Kepler-1788        sigma=0.63   n=2   ok
[  9/29] TOI-2133           sigma=0.05   n=5   ok
[ 10/29] Kepler-1733        sigma=0.06   n=3   ok
[ 11/29] Kepler-145         sigma=1.64   n=16  ok
[ 12/29] Kepler-1133        sigma=0.12   n=3   ok
[ 13/29] TOI-913            sigma=0.34   n=2   ok
[ 14/29] TOI-444            sigma=0.57   n=2   ok
[ 15/29] Kepler-1990        sigma=NaN    n=1   err_undef(n=1)
[ 16/29] EPIC 211822797     sigma=0.90   n=5   ok
[ 17/29] TOI-4640           sigma=1.02   n=7   ok
[ 18/29] POTS-1             sigma=0.09   n=2   ok
[ 19/29] TOI-4860           sigma=4.74   n=16  ok


Could not resolve "Qatar-10" to a sky position.
Could not resolve "Qatar-10" to a sky position.
Could not resolve "Qatar-10" to a sky position.
Could not resolve "Qatar-10" to a sky position.


[ 20/29] Qatar-10           sigma=NaN    n=0   no_lc
[ 21/29] K2-332             sigma=0.22   n=11  ok
[ 22/29] K2-85              sigma=0.30   n=3   ok
[ 23/29] TOI-5713           sigma=0.46   n=3   ok
[ 24/29] Kepler-1919        sigma=0.39   n=4   ok
[ 25/29] Kepler-10          sigma=0.09   n=2   ok
[ 26/29] K2-266             sigma=0.50   n=2   ok
[ 27/29] K2-191             sigma=0.92   n=6   ok
[ 28/29] K2-209             sigma=0.46   n=2   ok
[ 29/29] K2-282             sigma=0.97   n=4   ok

>> 已保存 -> vetting_result_100.csv
   通过: 25  不通过: 2  无法判定: 2


## 3. 校验二：行星半径合理性（批量 TIC 查询）

**提速关键**：识别目标名中的 TIC 编号，用 `Catalogs.query_criteria` 一次 SQL 查询拿回所有恒星参数，替代逐条 `query_object`。

In [5]:
from astroquery.mast import Catalogs

print('=== 校验二：行星半径 ===')

# ---- 辅助函数 ----
def _sf(x):
    try: return np.nan if x is None or np.ma.is_masked(x) else float(x)
    except: return np.nan

def _spectral(teff):
    if not np.isfinite(teff): return 'Unknown'
    if teff >= 30000: return 'O'
    if teff >= 10000: return 'B'
    if teff >= 7500:  return 'A'
    if teff >= 6000:  return 'F'
    if teff >= 5200:  return 'G'
    if teff >= 3700:  return 'K'
    if teff >= 2400:  return 'M'
    return 'L/T'

# ---- 分离两类目标：可批量查 TIC ID 的 vs 需按名查的 ----
STAR_RAD, STAR_TEFF, STAR_TIC = {}, {}, {}

tic_ids = []
name_targets = []
for _, row in df.iterrows():
    t = str(row['Target']).strip()
    if t.upper().startswith('TIC') and t[3:].strip().isdigit():
        tic_ids.append(int(t[3:].strip()))
    else:
        name_targets.append(t)

print(f'  TIC 编号目标(可批量): {len(tic_ids)}  名称目标(逐个): {len(name_targets)}')

# ---- 批量查 TIC ----
if tic_ids:
    ids_str = ','.join(str(x) for x in tic_ids)
    try:
        batch = Catalogs.query_criteria(
            catalog='TIC', select='ID,rad,Teff,mass,Tmag',
            where=f'ID IN ({ids_str})')
        if batch is not None:
            for r in batch:
                key = f'TIC {int(r["ID"])}'
                STAR_RAD[key] = _sf(r['rad'])
                STAR_TEFF[key] = _sf(r['Teff'])
                STAR_TIC[key] = int(r['ID'])
            print(f'  批量查询命中: {len(STAR_RAD)} 个')
    except Exception as e:
        print(f'  批量查询失败: {e}，将逐条查询...')
        for tid in tic_ids:
            try:
                cat = Catalogs.query_object(f'TIC {tid}', catalog='TIC', radius=0.02)
                if cat is not None and len(cat) > 0:
                    r_ = cat[0]
                    STAR_RAD[f'TIC {tid}'] = _sf(r_['rad'])
                    STAR_TEFF[f'TIC {tid}'] = _sf(r_['Teff'])
                    STAR_TIC[f'TIC {tid}'] = int(r_['ID'])
            except Exception:
                pass

# ---- 按名称逐个查 (少数) ----
for t in name_targets:
    try:
        cat = Catalogs.query_object(t, catalog='TIC', radius=0.02)
        if cat is not None and len(cat) > 0:
            try:
                order = np.argsort([_sf(v) for v in cat['Tmag']])
                cat = cat[order]
            except Exception:
                pass
            r = cat[0]
            STAR_RAD[t] = _sf(r['rad'])
            STAR_TEFF[t] = _sf(r['Teff'])
            STAR_TIC[t] = int(r['ID'])
    except Exception:
        pass

print(f'  总计查询到恒星参数: {len(STAR_RAD)} 个')

=== 校验二：行星半径 ===
  TIC 编号目标(可批量): 0  名称目标(逐个): 29
  总计查询到恒星参数: 28 个


In [6]:
# ---- 逐目标判定半径合理性 ----
rj_list, re_list, spec_list, rsn_list = [], [], [], []
tic_list, rstar_list, teff_list = [], [], []
plausible_list = []

for i, row in df.iterrows():
    t = str(row['Target']).strip()
    rad = STAR_RAD.get(t, np.nan)
    te = STAR_TEFF.get(t, np.nan)
    ti = STAR_TIC.get(t, None)
    sp = _spectral(te)
    d = float(row['Depth (ppm)']) * 1e-6

    if not np.isfinite(d) or d <= 0:
        ok, rj, rre, reason = pd.NA, np.nan, np.nan, 'bad_depth'
    elif not np.isfinite(rad):
        ok, rj, rre, reason = pd.NA, np.nan, np.nan, 'no_stellar_radius'
    else:
        rj = np.sqrt(d) * rad * R_SUN_TO_RJUP
        rre = rj * R_JUP_TO_REARTH
        mx = MAX_RP_MDWARF if sp == 'M' else MAX_RP_GENERAL
        if rj <= mx:
            ok, reason = True, 'plausible'
        else:
            ok, reason = False, ('too_large_mdwarf' if sp == 'M' else 'too_large')

    rj_list.append(rj); re_list.append(rre); spec_list.append(sp)
    rsn_list.append(reason); tic_list.append(ti)
    rstar_list.append(rad); teff_list.append(te)
    plausible_list.append(ok)

    rt = 'NaN' if not np.isfinite(rj) else f'{rj:.2f}'
    rs = 'NA' if not np.isfinite(rad) else f'{rad:.3f}'
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} "
          f"R*={rs:<7} 型={sp:<5} Rp={rt:<7}Rjup  {str(ok):<5}")

df['TIC_ID'] = tic_list
df['Stellar_Radius_Rsun'] = rstar_list
df['Teff_K'] = teff_list
df['Spectral_Type'] = spec_list
df['Rp_Rjup'] = rj_list
df['Rp_Rearth'] = re_list
df['radius_reason'] = rsn_list
df['planet_radius_plausible'] = pd.array(plausible_list, dtype='boolean')

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   合理: {int((df["planet_radius_plausible"]==True).sum())}  '
      f'不合理: {int((df["planet_radius_plausible"]==False).sum())}  '
      f'无法判定: {int(df["planet_radius_plausible"].isna().sum())}')

[  1/29] Kepler-528         R*=12.822  型=K     Rp=12.50  Rjup  False
[  2/29] TOI-5076           R*=0.850   型=K     Rp=0.17   Rjup  True 
[  3/29] TOI-2580           R*=1.829   型=F     Rp=0.95   Rjup  True 
[  4/29] Kepler-1254        R*=0.719   型=K     Rp=0.83   Rjup  True 
[  5/29] TOI-1410           R*=0.779   型=K     Rp=0.25   Rjup  True 
[  6/29] Kepler-224         R*=1.493   型=F     Rp=2.25   Rjup  True 
[  7/29] Kepler-1546        R*=44.916  型=K     Rp=53.67  Rjup  False
[  8/29] Kepler-1788        R*=0.861   型=G     Rp=1.23   Rjup  True 
[  9/29] TOI-2133           R*=0.716   型=K     Rp=0.20   Rjup  True 
[ 10/29] Kepler-1733        R*=1.639   型=F     Rp=2.23   Rjup  True 
[ 11/29] Kepler-145         R*=1.249   型=G     Rp=0.29   Rjup  True 
[ 12/29] Kepler-1133        R*=2.273   型=F     Rp=2.76   Rjup  False
[ 13/29] TOI-913            R*=0.733   型=K     Rp=0.16   Rjup  True 
[ 14/29] TOI-444            R*=0.794   型=K     Rp=0.18   Rjup  True 
[ 15/29] Kepler-1990        R*=10.

## 4. 校验三：NASA 档案交叉比对

In [7]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

print('=== 校验三：NASA 档案交叉比对 ===')

_CACHE = {}

def _okt(x):
    try: return x is not None and not pd.isna(x) and int(x) > 0
    except: return False

def crossmatch(target, tic_id=None):
    '''返回 (已知/新候选, 命中来源)。'''
    key = (str(target), int(tic_id) if _okt(tic_id) else None)
    if key in _CACHE:
        return _CACHE[key]
    label, source = '新候选', 'none'
    # (1) 确认行星表：按名称
    try:
        t = NasaExoplanetArchive.query_object(str(target))
        if t is not None and len(t) > 0:
            label, source = '已知', 'confirmed'
    except Exception:
        source = 'query_error'
    # (2) TOI 表：按 TIC ID
    if label == '新候选' and _okt(tic_id):
        try:
            q = NasaExoplanetArchive.query_criteria(
                table='toi', select='toi,tid,tfopwg_disp',
                where=f'tid={int(tic_id)}')
            if q is not None and len(q) > 0:
                label, source = '已知', 'TOI'
        except Exception:
            if source != 'confirmed':
                source = 'query_error'
    _CACHE[key] = (label, source)
    return (label, source)

labels, sources = [], []
has_tic = 'TIC_ID' in df.columns
for i, row in df.iterrows():
    tic = row['TIC_ID'] if has_tic else None
    lb, so = crossmatch(row['Target'], tic)
    labels.append(lb); sources.append(so)
    print(f"[{i+1:>3}/{len(df)}] {str(row['Target'])[:18]:<18} -> {lb:<6} ({so})")

df['已知/新候选'] = labels
df['match_source'] = sources

# ---- 即时写入 CSV ----
existing = [c for c in FINAL_COLS if c in df.columns]
df[existing].to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'\n>> 已保存 -> {OUTPUT_CSV}')
print(f'   已知: {labels.count("已知")}  新候选: {labels.count("新候选")}')

=== 校验三：NASA 档案交叉比对 ===
[  1/29] Kepler-528         -> 已知     (confirmed)
[  2/29] TOI-5076           -> 已知     (confirmed)
[  3/29] TOI-2580           -> 已知     (confirmed)
[  4/29] Kepler-1254        -> 已知     (confirmed)
[  5/29] TOI-1410           -> 已知     (confirmed)
[  6/29] Kepler-224         -> 已知     (confirmed)
[  7/29] Kepler-1546        -> 已知     (confirmed)
[  8/29] Kepler-1788        -> 已知     (confirmed)
[  9/29] TOI-2133           -> 已知     (confirmed)
[ 10/29] Kepler-1733        -> 已知     (confirmed)
[ 11/29] Kepler-145         -> 已知     (confirmed)
[ 12/29] Kepler-1133        -> 已知     (confirmed)
[ 13/29] TOI-913            -> 已知     (confirmed)
[ 14/29] TOI-444            -> 已知     (confirmed)
[ 15/29] Kepler-1990        -> 已知     (confirmed)
[ 16/29] EPIC 211822797     -> 已知     (confirmed)
[ 17/29] TOI-4640           -> 已知     (confirmed)
[ 18/29] POTS-1             -> 已知     (confirmed)
[ 19/29] TOI-4860           -> 已知     (confirmed)
[ 20/29] Qatar-10         

## 5. 最终汇总与高置信度清单

In [8]:
# ---- Person1 数据完整性校验 ----
_p = [c for c in P1_COLS if c in df.columns]
assert df[_p].equals(ORIG_DF[_p]), 'Person1 原始数据被意外修改!'
print('✓ Person1 原始数据完整性校验通过\n')

# ---- 高置信度：奇偶通过 + 半径合理 + 非 Person1 原判假阳性 ----
fp = df['False Positive'].astype(str).str.lower().isin(['true', '1'])
mask = (df['odd_even_pass'] == True) & (df['planet_radius_plausible'] == True) & (~fp)
hc = df[mask.values].copy()

# ---- 按 FINAL_COLS 列序写入最终结果 ----
final = df[[c for c in FINAL_COLS if c in df.columns]].copy()
final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
hc.to_csv('high_confidence_candidates.csv', index=False, encoding='utf-8-sig')

print('=' * 64)
print('三项质控汇总')
print('=' * 64)
print(f'候选总数     : {len(df)}')
print(f'奇偶通过     : {int((df["odd_even_pass"]==True).sum())}')
print(f'半径合理     : {int((df["planet_radius_plausible"]==True).sum())}')
print(f'已知/新候选   : {labels.count("已知")} / {labels.count("新候选")}')
print(f'>> 高置信度  : {len(hc)}  ->  high_confidence_candidates.csv')
print('=' * 64)
final.head(10)

✓ Person1 原始数据完整性校验通过

三项质控汇总
候选总数     : 29
奇偶通过     : 25
半径合理     : 23
已知/新候选   : 29 / 0
>> 高置信度  : 3  ->  high_confidence_candidates.csv


,Target,Sector,Period (d),Period Err (d),Depth (ppm),Duration (hr),SNR,False Positive,depth_odd_ppm,depth_even_ppm,n_transits,odd_even_sigma,odd_even_reason,odd_even_pass,TIC_ID,Stellar_Radius_Rsun,Teff_K,Spectral_Type,Rp_Rjup,Rp_Rearth,radius_reason,planet_radius_plausible,已知/新候选,match_source
0,Kepler-528,40,8.25305,0.21041,10031.3,1.10,5.58,True,12167.045425,7787.516274,3,1.113891,ok,True,271877640.0,12.821900,4566.10,K,12.496631,140.073482,too_large,False,已知,confirmed
1,TOI-5076,70,5.58267,0.13242,439.2,0.50,3.73,True,388.530389,389.275169,4,0.004146,ok,True,303432813.0,0.850313,4794.37,K,0.173409,1.943725,plausible,True,已知,confirmed
2,TOI-2580,59,3.39686,0.00085,2841.2,4.15,48.09,False,2338.707519,3115.070382,7,6.462576,ok,False,102713734.0,1.829040,6590.00,F,0.948716,10.634058,plausible,True,已知,confirmed
3,Kepler-1254,80,11.90309,0.31842,14136.2,1.70,5.59,True,22918.020924,14694.819697,2,1.345499,ok,True,48506710.0,0.718755,4898.13,K,0.831590,9.321207,plausible,True,已知,confirmed
4,TOI-1410,16,1.21631,0.00011,1082.3,1.10,21.18,False,1065.115691,1156.406971,16,0.856702,ok,True,199444169.0,0.778749,4507.00,K,0.249306,2.794449,plausible,True,已知,confirmed
5,Kepler-224,80,6.42276,0.16333,23983.7,0.50,4.83,True,31830.352045,21008.397775,5,1.113407,ok,True,159642170.0,1.493250,6106.00,F,2.250363,25.224092,plausible,True,已知,confirmed
6,Kepler-1546,41,4.29489,0.11983,15075.4,1.10,4.94,True,21128.187214,9905.524446,6,1.979628,ok,True,169466741.0,44.916500,4174.00,K,53.666376,601.541037,too_large,False,已知,confirmed
7,Kepler-1788,40,14.62087,0.40759,21541.6,1.70,6.75,True,23073.357828,19383.876054,2,0.625044,ok,True,159646443.0,0.860579,5792.04,G,1.229113,13.777000,plausible,True,已知,confirmed
8,TOI-2133,25,3.82612,0.04872,809.8,1.70,6.95,True,812.025126,823.138055,5,0.049359,ok,True,39200363.0,0.716312,4175.00,K,0.198360,2.223392,plausible,True,已知,confirmed
9,Kepler-1733,74,7.07060,0.16277,19574.2,0.50,3.51,True,17039.571131,17587.860677,3,0.061833,ok,True,122781772.0,1.638800,6611.00,F,2.231156,25.008802,plausible,True,已知,confirmed
